# Explore: Anomaly Detection for Spending

This notebook demonstrates the spending agent's detection algorithms.
We generate synthetic transaction data with planted anomalies, then
run Z-score and Isolation Forest to see if they catch them.

**What you'll learn:**
1. How Z-score identifies statistical outliers
2. How Isolation Forest works visually
3. Why combining methods gives better results
4. How to tune detection sensitivity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest

np.random.seed(42)  # Reproducible results
print('All imports successful!')

## Step 1: Generate Synthetic Transactions

We create realistic-looking spending data with known anomalies
planted at specific positions.

In [ ]:
# Normal spending patterns (mean, std)
categories = {
    'Groceries': (85, 15),     # Average $85, std $15
    'Transport': (50, 10),
    'Food': (30, 8),
    'Entertainment': (35, 12),
    'Utilities': (120, 20),
}

transactions = []
for category, (mean, std) in categories.items():
    # Generate 20 normal transactions per category
    for i in range(20):
        amount = max(5, np.random.normal(mean, std))  # No negative amounts
        transactions.append({
            'id': len(transactions),
            'category': category,
            'amount': round(amount, 2),
            'is_anomaly': False,
        })

# Plant anomalies
anomalies = [
    {'id': 200, 'category': 'Groceries', 'amount': 500.00, 'is_anomaly': True},
    {'id': 201, 'category': 'Food', 'amount': 250.00, 'is_anomaly': True},
    {'id': 202, 'category': 'Entertainment', 'amount': 400.00, 'is_anomaly': True},
    {'id': 203, 'category': 'Transport', 'amount': 300.00, 'is_anomaly': True},
]
transactions.extend(anomalies)

df = pd.DataFrame(transactions)
print(f'Total transactions: {len(df)}')
print(f'Planted anomalies: {len(anomalies)}')
print(f'\nAmount stats by category:')
df.groupby('category')['amount'].describe()[['mean', 'std', 'min', 'max']].round(2)

## Step 2: Visualize the Data

Box plots show the distribution per category. Anomalies should
appear as dots far outside the whiskers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Box plot by category
categories_list = list(categories.keys())
data_by_cat = [df[df['category'] == cat]['amount'].values for cat in categories_list]
bp = ax.boxplot(data_by_cat, labels=categories_list, patch_artist=True)

# Color the boxes
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

# Highlight anomalies
for _, row in df[df['is_anomaly']].iterrows():
    cat_idx = categories_list.index(row['category']) + 1
    ax.plot(cat_idx, row['amount'], 'r*', markersize=15, label='Planted anomaly' if _ == anomalies[0]['id'] else '')

ax.set_ylabel('Amount ($)')
ax.set_title('Transaction Amounts by Category (stars = planted anomalies)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: Z-Score Detection

Z-score measures how many standard deviations a value is from the mean.

```
z = (x - mean) / std
```

Convention: |z| > 2 is unusual, |z| > 3 is very unusual.

In [ ]:
# Calculate Z-scores per category
df['z_score'] = 0.0

for category, group in df.groupby('category'):
    z_scores = np.abs(stats.zscore(group['amount']))
    df.loc[group.index, 'z_score'] = z_scores

# Flag anomalies
df['z_flagged'] = df['z_score'] > 2

print('=== Z-Score Results ===')
z_flagged = df[df['z_flagged']].sort_values('z_score', ascending=False)
print(f'Flagged {len(z_flagged)} transactions:\n')
print(z_flagged[['id', 'category', 'amount', 'z_score', 'is_anomaly']].to_string(index=False))

# Did we catch all planted anomalies?
planted_caught = z_flagged[z_flagged['is_anomaly']]
print(f'\nPlanted anomalies caught: {len(planted_caught)}/{len(anomalies)}')

## Step 4: Isolation Forest Detection

Isolation Forest randomly splits data. Anomalies need fewer splits
to be isolated because they're far from the crowd.

In [ ]:
# Prepare features: amount + encoded category
features = df[['amount']].copy()
features['category_code'] = pd.Categorical(df['category']).codes

# Fit Isolation Forest
iso = IsolationForest(
    contamination=0.1,  # Expect ~10% anomalies
    random_state=42,
    n_estimators=100,
)

df['iso_prediction'] = iso.fit_predict(features)
df['iso_flagged'] = df['iso_prediction'] == -1  # -1 = anomaly
df['iso_score'] = iso.decision_function(features)  # Lower = more anomalous

print('=== Isolation Forest Results ===')
iso_flagged = df[df['iso_flagged']].sort_values('iso_score')
print(f'Flagged {len(iso_flagged)} transactions:\n')
print(iso_flagged[['id', 'category', 'amount', 'iso_score', 'is_anomaly']].to_string(index=False))

planted_caught = iso_flagged[iso_flagged['is_anomaly']]
print(f'\nPlanted anomalies caught: {len(planted_caught)}/{len(anomalies)}')

## Step 5: Comparing Both Methods

When both methods agree, we have high confidence it's a real anomaly.

In [ ]:
df['both_flagged'] = df['z_flagged'] & df['iso_flagged']

# Build comparison table
comparison = pd.DataFrame({
    'Z-Score Only': [len(df[df['z_flagged'] & ~df['iso_flagged']])],
    'Isolation Forest Only': [len(df[~df['z_flagged'] & df['iso_flagged']])],
    'Both Methods': [len(df[df['both_flagged']])],
    'Neither': [len(df[~df['z_flagged'] & ~df['iso_flagged']])],
})
print('Detection Overlap:')
print(comparison.to_string(index=False))

# How many planted anomalies did each method catch?
planted = df[df['is_anomaly']]
print(f'\n--- Planted Anomaly Detection Rate ---')
print(f'Z-Score caught:          {planted["z_flagged"].sum()}/{len(planted)}')
print(f'Isolation Forest caught:  {planted["iso_flagged"].sum()}/{len(planted)}')
print(f'Both methods caught:     {planted["both_flagged"].sum()}/{len(planted)}')

## Step 6: Visualize Detection Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Z-score distribution
ax1.hist(df['z_score'], bins=30, alpha=0.7, color='steelblue')
ax1.axvline(x=2, color='orange', linestyle='--', label='Threshold (z=2)')
ax1.axvline(x=3, color='red', linestyle='--', label='Threshold (z=3)')
ax1.set_xlabel('Z-Score')
ax1.set_ylabel('Count')
ax1.set_title('Z-Score Distribution')
ax1.legend()

# Plot 2: Isolation Forest anomaly scores
colors = ['red' if f else 'steelblue' for f in df['iso_flagged']]
ax2.scatter(range(len(df)), df['iso_score'], c=colors, alpha=0.5, s=20)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax2.set_xlabel('Transaction Index')
ax2.set_ylabel('Anomaly Score (lower = more anomalous)')
ax2.set_title('Isolation Forest Scores (red = flagged)')

plt.tight_layout()
plt.show()

## Exercises for You

1. **Change contamination**: Try 0.05 and 0.2. How does the number of flags change?
2. **Subtler anomalies**: Change the planted anomaly amounts to be only 2x the mean instead of 5x. Can the methods still detect them?
3. **Add time features**: Create a `day_of_month` column. Can Isolation Forest catch a pattern like "unusually high spending at end of month"?
4. **Try LOF**: Replace IsolationForest with `sklearn.neighbors.LocalOutlierFactor`. Compare results.